In [3]:
"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": [x,y,z],
        "right_elbow": [x,y,z],
        "right_wrist": [x,y,z],
        "right_hip": [x,y,z]
    },
    1: { # Frame 1
        "right_shoulder": [0.52, 0.50, 0.50],
        "right_elbow": [0.65, 0.20, 0.55],
        "right_wrist": [0.75, 0.05, 0.60],
        "right_hip": [0.51, 0.80, 0.50]
    },
    # ...
}
"""
from typing import Any
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np


def round_landmark(landmark):
    return [round(landmark.x, 3), round(landmark.y, 3), round(landmark.z, 3)]


model_path = "../models/pose_landmarker_heavy.task"
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload: dict[int, Any] = {}
frame_idx = 0
# Open the webcam
cap = cv2.VideoCapture("../cache/raw_videos/test_clear_trim3.mp4")
fps = round(cap.get(cv2.CAP_PROP_FPS), 3)
print(f"Frames per second: {fps}")
while True:
    # Process the video frames
    success, frame = cap.read()
    if not success:
        print("End of video reached or failed to read the video frame.")
        break

    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    # set up for timestamp in milliseconds for the current frame
    timestamp_ms = int((frame_idx / fps) * 1000)

    detection_result = detector.detect_for_video(mp_image, timestamp_ms=timestamp_ms)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        # Note: detection_result.pose_landmarks is a list of PoseLandmarkList, where each PoseLandmarkList corresponds to a detected person in the frame. For simplicity, we will only consider the first detected person (if multiple people are detected).
        landmarks = detection_result.pose_landmarks[0]

        target_indices = [11, 12, 15, 16, 23, 24, 25, 26, 27, 28]

        frame_coords = [round_landmark(landmarks[i]) for i in target_indices]
        frame_matrix = np.array(frame_coords)
        print(frame_matrix)
        left_hip = frame_matrix[4]
        right_hip = [5]

        midpoint = frame_matrix[[4, 5]].mean(axis=0)
        normalized_coords = frame_matrix - midpoint

        pose_data_payload[frame_idx] = normalized_coords
    else:
        pose_data_payload[frame_idx] = np.zeros(
            (10, 3)
        )  # Use a list of zeros for frames with no detected landmarks
        print("No pose landmarks detected.")
    frame_idx += 1
print("pose_data_payload:", pose_data_payload)
cap.release()
detector.close()


libEGL warning: DRI3 error: Could not get DRI3 device
libEGL warning: Ensure your X server supports DRI3 to get accelerated rendering
I0000 00:00:1779554984.466482   14417 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779554984.524951   14467 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779554984.606551   14424 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779554984.748014   14425 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Frames per second: 60.026


W0000 00:00:1779554985.067308   14418 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Pose landmarks detected:
[[ 0.513  0.422  0.048]
 [ 0.577  0.449 -0.007]
 [ 0.497  0.33   0.228]
 [ 0.624  0.415 -0.051]
 [ 0.537  0.64   0.021]
 [ 0.577  0.643 -0.021]
 [ 0.511  0.762  0.041]
 [ 0.594  0.774 -0.011]
 [ 0.49   0.891  0.007]
 [ 0.588  0.915 -0.044]]
Pose landmarks detected:
[[ 0.512  0.422  0.047]
 [ 0.575  0.449  0.005]
 [ 0.491  0.33   0.229]
 [ 0.621  0.412 -0.017]
 [ 0.536  0.639  0.018]
 [ 0.576  0.642 -0.018]
 [ 0.51   0.762  0.041]
 [ 0.594  0.773 -0.005]
 [ 0.49   0.891  0.013]
 [ 0.589  0.914 -0.022]]
Pose landmarks detected:
[[ 0.509  0.422  0.046]
 [ 0.573  0.448  0.014]
 [ 0.482  0.333  0.208]
 [ 0.615  0.405  0.013]
 [ 0.534  0.639  0.016]
 [ 0.574  0.641 -0.016]
 [ 0.508  0.762  0.038]
 [ 0.593  0.771 -0.003]
 [ 0.49   0.892  0.013]
 [ 0.59   0.91  -0.015]]
Pose landmarks detected:
[[ 0.507  0.423  0.044]
 [ 0.572  0.448  0.014]
 [ 0.479  0.334  0.205]
 [ 0.612  0.402 -0.101]
 [ 0.533  0.639  0.013]
 [ 0.573  0.641 -0.014]
 [ 0.506  0.762  0.036]
 [ 0.592 

In [4]:
import numpy as np


video_tensor = np.array(list(pose_data_payload.values()))  # Convert the pose data payload to a NumPy array
# print("video_tensor shape:", video_tensor)
total_frames = video_tensor.shape[0]
print("Total frames processed:", total_frames)
print("video_tensor shape:", video_tensor.shape)

video_tensor_flat = video_tensor.reshape(total_frames, -1)  # Flatten the last two dimensions
print("video_tensor_flat shape:", video_tensor_flat.shape)
print("video_tensor_flat:", video_tensor_flat)
SQE_LEN = 11
window = []
for i in range(total_frames - SQE_LEN + 1):
    window.append(video_tensor_flat[i : i + SQE_LEN])

final_input_sensor = np.array(window)
print("final_input_sensor shape:", final_input_sensor.shape)
print("final_input_sensor:", final_input_sensor)

Total frames processed: 123
video_tensor shape: (123, 10, 3)
video_tensor_flat shape: (123, 30)
video_tensor_flat: [[-0.044  -0.2195  0.048  ...  0.031   0.2735 -0.044 ]
 [-0.044  -0.2185  0.047  ...  0.033   0.2735 -0.022 ]
 [-0.045  -0.218   0.046  ...  0.036   0.27   -0.015 ]
 ...
 [-0.0365 -0.214  -0.008  ...  0.0185  0.287   0.126 ]
 [-0.037  -0.214  -0.008  ...  0.018   0.285   0.133 ]
 [-0.037  -0.213  -0.0085 ...  0.018   0.286   0.1295]]
final_input_sensor shape: (113, 11, 30)
final_input_sensor: [[[-0.044  -0.2195  0.048  ...  0.031   0.2735 -0.044 ]
  [-0.044  -0.2185  0.047  ...  0.033   0.2735 -0.022 ]
  [-0.045  -0.218   0.046  ...  0.036   0.27   -0.015 ]
  ...
  [-0.0505 -0.205   0.014  ...  0.0475  0.265  -0.011 ]
  [-0.0545 -0.1995  0.013  ...  0.0485  0.2635 -0.036 ]
  [-0.0565 -0.197   0.015  ...  0.0495  0.263  -0.044 ]]

 [[-0.044  -0.2185  0.047  ...  0.033   0.2735 -0.022 ]
  [-0.045  -0.218   0.046  ...  0.036   0.27   -0.015 ]
  [-0.046  -0.217   0.0445 ...  0

In [7]:
import torch
import torch.nn as nn

class BadmintonMovementLSTM(nn.Module):
    def __init__(self):
        super(BadmintonMovementLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=30,  # 10 landmarks * 3 coordinates each
            hidden_size=64,
            batch_first=True
        )
        
        self.dropout = nn.Dropout(0.3)
        
        self.fc = nn.Linear(
            in_features=64,
            out_features=1
        )
        
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        """x shape: (batch_size, sequence_length=11, input_size=30)"""
        lstm_out, _ = self.lstm(x) # ht=LSTM(xt,ht-1)
        
        last_time_step_out = lstm_out[:, -1, :] # Get the output of the last time step
        
        dropout_out = self.dropout(last_time_step_out)
        
        logits = self.fc(dropout_out)
        
        probability = self.sigmoid(logits)
        
        return probability
        
        
        
model = BadmintonMovementLSTM()

# print(model)
dummy_input = torch.tensor(final_input_sensor, dtype=torch.float32)  # Convert the final input sensor to a PyTorch tensor

print("dummy_input shape:", dummy_input.shape)
prediction = model(dummy_input)
print("prediction shape:", prediction.shape)
print("prediction:", prediction[:5])  # Print the first 5 predictions

dummy_input shape: torch.Size([113, 11, 30])
prediction shape: torch.Size([113, 1])
prediction: tensor([[0.4971],
        [0.5066],
        [0.4995],
        [0.4993],
        [0.5024]], grad_fn=<SliceBackward0>)
